# SysQ Live Trading Simulation

This notebook demonstrates the **"Reality-First"** Live Trading Workflow.

### Workflow
1.  **Initialize Real Account**: Create a SQLite database to track real assets.
2.  **Sync Broker State**: Input the initial state (Cash + Positions) from your broker.
3.  **Load Model**: Load a pre-trained model (trained on historical data).
4.  **Run Daily Plan**: Generate buy/sell orders for the next trading day.
5.  **Simulate Execution**: Manually update the account to reflect "filled" orders.
6.  **Loop**: Move to the next day.

In [ ]:
import sys
import os
import pandas as pd
from qsys.live.account import RealAccount
from qsys.live.manager import LiveManager
from qsys.data.adapter import QlibAdapter

# Initialize Qlib (Required for data fetching)
adapter = QlibAdapter()
adapter.init_qlib()

# Setup Paths
DB_PATH = "data/live_simulation.db"
MODEL_PATH = "data/models/lgb_alpha158_20250201_160359" # Replace with your actual model path if different
# Ensure the model exists, or run tutorial.ipynb first to generate it.
# If you don't have a model, we can't predict, but we can demo the account sync.

## Step 1: Initialize Real Account
We create a persistent `RealAccount` backed by SQLite. This represents your **Broker Account**.

In [ ]:
# Create/Reset Account DB
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

account = RealAccount(db_path=DB_PATH)
print(f"Initialized DB at {DB_PATH}")

## Step 2: Sync Broker State (Day T)
Imagine today is **2026-02-01**. You have 1,000,000 Cash and NO positions.

In [ ]:
current_date = "2026-02-01"

# Initial State
initial_cash = 1_000_000.0
initial_positions = pd.DataFrame(columns=['symbol', 'amount', 'price', 'cost_basis'])

# Sync to DB
account.sync_broker_state(
    date=current_date,
    cash=initial_cash,
    positions=initial_positions
)

# Verify
state = account.get_state()
print("Current State:", state)

## Step 3: Generate Plan for Tomorrow
We want to trade on **2022-11-01** (Using historical date where we have data for demo).
Wait, let's use a date where we actually have data in our Qlib bin.
Our tutorial data is usually 2022. Let's pretend today is **2022-11-01**.

In [ ]:
target_date = "2022-11-01"

# First, we need to sync the account as if it's the night before target_date
account.sync_broker_state(
    date="2022-10-31",
    cash=1_000_000.0,
    positions=pd.DataFrame(columns=['symbol', 'amount', 'price', 'cost_basis'])
)

# Init Manager
manager = LiveManager(model_path=MODEL_PATH, db_path=DB_PATH)

# Run Plan
# This will:
# 1. Fetch features for 2022-11-01 (or latest available)
# 2. Predict
# 3. Compare with Account State (Cash only)
# 4. Generate Buys
plan = manager.run_daily_plan(target_date)

if plan is not None:
    print("\nGenerated Plan:")
    print(plan.head())

## Step 4: Simulate Execution (Manual Sync)
You sent the orders to the broker. The next day (**2022-11-02**), you check your account.
Some bought, some didn't. Price changed.

Let's assume we bought `SZ000001` (Fake example).

In [ ]:
next_date = "2022-11-02"

# Define what we actually hold now
new_positions = pd.DataFrame([
    {'symbol': 'SZ000001', 'amount': 1000, 'price': 12.5, 'cost_basis': 12.0},
    {'symbol': 'SH600000', 'amount': 2000, 'price': 8.5, 'cost_basis': 8.4}
])

# Cash reduced
new_cash = 1_000_000 - (1000*12.0) - (2000*8.4)

# Sync Reality
account.sync_broker_state(
    date=next_date,
    cash=new_cash,
    positions=new_positions
)

# Check State
print(account.get_state(next_date))

## Step 5: Next Day Plan
Now the system knows we hold these stocks. It will generate a plan to adjust them.

In [ ]:
# Run plan for next day
plan_next = manager.run_daily_plan(next_date)

# Notice how it considers current positions!
if plan_next is not None:
    print(plan_next)